In [1]:
%env DATA_PATH=../../../data
from db import *
from api.routers.rolls import get_roll
from sqlalchemy import select
from sqlalchemy.orm import selectinload
from lib.paths import resolve_path
from lib.fit import load_fit_file, get_gps_data, estimate_fit_timestamp, get_fit_graph_data, get_sensor_data, get_camera_starts, get_camera_ends
from lib.racebox import get_racebox_graph_data
from lib.signal import lowpass_filter
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import timedelta
import gtsam
import pymap3d as pm
from tqdm.notebook import tqdm


DATA_PATH = '../../../data'

env: DATA_PATH=../../../data


In [2]:
session = Session(engine)
query = select(Roll).join(RollFile).join(File).where(File.type == 'racebox').where(Roll.id.in_(
  select(Roll.id).join(RollFile).join(File).where(File.type == 'fit')
))
rolls = session.execute(query).scalars().all()
rolls

[Roll(id=44, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=3),
 Roll(id=1388, roll_date_id=154, buggy_id=3, driver_id=3, roll_number=6),
 Roll(id=1387, roll_date_id=154, buggy_id=4, driver_id=2, roll_number=7),
 Roll(id=37, roll_date_id=6, buggy_id=1, driver_id=5, roll_number=1),
 Roll(id=38, roll_date_id=6, buggy_id=1, driver_id=5, roll_number=2),
 Roll(id=39, roll_date_id=7, buggy_id=3, driver_id=3, roll_number=3),
 Roll(id=1401, roll_date_id=155, buggy_id=3, driver_id=3, roll_number=4),
 Roll(id=45, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=4)]

In [3]:
files = [{rf.file.type: rf for rf in roll.roll_files} | {'roll': roll} for roll in rolls]

In [4]:
roll_data = []
for roll in files:
  roll_data.append((roll['roll'], roll['fit'], roll['racebox'], False))
  if 'fit_c' in roll:
    roll_data.append((roll['roll'], roll['fit_c'], roll['racebox'], True))
roll_data

[(Roll(id=44, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=3),
  RollFile(id=89, roll_id=44, file_id=37, local_start_ms=3488, local_end_ms=262595),
  RollFile(id=3444, roll_id=44, file_id=3169, local_start_ms=None, local_end_ms=None),
  False),
 (Roll(id=1388, roll_date_id=154, buggy_id=3, driver_id=3, roll_number=6),
  RollFile(id=3255, roll_id=1388, file_id=3062, local_start_ms=3315, local_end_ms=260653),
  RollFile(id=3450, roll_id=1388, file_id=3174, local_start_ms=None, local_end_ms=None),
  False),
 (Roll(id=1387, roll_date_id=154, buggy_id=4, driver_id=2, roll_number=7),
  RollFile(id=3253, roll_id=1387, file_id=3060, local_start_ms=3457, local_end_ms=204606),
  RollFile(id=3451, roll_id=1387, file_id=3175, local_start_ms=None, local_end_ms=None),
  False),
 (Roll(id=37, roll_date_id=6, buggy_id=1, driver_id=5, roll_number=1),
  RollFile(id=3312, roll_id=37, file_id=3056, local_start_ms=None, local_end_ms=None),
  RollFile(id=3452, roll_id=37, file_id=3176, local_start_m

In [5]:
roll, fit_file, racebox_file, is_c = roll_data[5]
print(roll.id, is_c)
fit = load_fit_file(resolve_path(fit_file.file.uri))
fit_gps = get_gps_data(fit)
fit_graphs = get_fit_graph_data(fit)
racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])

fit_start_time = estimate_fit_timestamp(fit)
racebox_start_time = racebox_file.file.start_time
print(fit_start_time, racebox_start_time)

fit_record_speed = pd.DataFrame.from_records(fit['record_mesgs']).set_index('timestamp')
fit_record_speed.index *= 1000

fig = go.Figure()
fig.add_trace(go.Scatter(x=racebox_graph_data['gps_data'].index + ((racebox_start_time - fit_start_time).total_seconds()) * 1000, y=racebox_graph_data['gps_data'].speed, mode='lines', name='racebox'))
fig.add_trace(go.Scatter(x=fit_gps.index, y=fit_gps.speed, mode='lines', name='fit'))
# fig.add_trace(go.Scatter(x=fit_record_speed.index, y=fit_record_speed.enhanced_speed, mode='lines', name='fit record'))
from lib.gpx import calculate_speed
# calced_speed = calculate_speed(fit_gps)
# fig.add_trace(go.Scatter(x=calced_speed.index, y=calced_speed, mode='lines', name='calculated speed'))
fig.show()

38 False
2026-03-14 12:15:46.877000 2026-03-14 12:16:05.040000


In [8]:
px.line(racebox_graph_data['gps_data'].elevation)

0/m, 1, 2, 3m, 4*, (5m, 6*), 8/, 9*m, 10

## Load

In [8]:
from gtsam.symbol_shorthand import X, V, B, R, O
from pymap3d import geodetic2enu
from lib.geo import load_elevation_data
from scipy.interpolate import RectBivariateSpline
from pyproj import Transformer
lat0, lon0, alt0 = 40.44163016, -79.94165829, 288.42151354


In [9]:
def make_elevation_spline(elevation, lat0, lon0, alt0):
    data = elevation.read(1)
    nrows, ncols = data.shape

    transform = elevation.transform
    col_coords = np.array([transform.c + (c + 0.5) * transform.a for c in range(ncols)])
    row_coords = np.array([transform.f + (r + 0.5) * transform.e for r in range(nrows)])

    to_wgs84 = Transformer.from_crs(elevation.crs, "EPSG:4326", always_xy=True)

    col_mesh, row_mesh = np.meshgrid(col_coords, row_coords)
    lon_grid, lat_grid = to_wgs84.transform(col_mesh, row_mesh)

    east_grid, north_grid, _ = geodetic2enu(
        lat_grid, lon_grid, np.zeros_like(lat_grid), lat0, lon0, alt0
    )

    east_axis  = east_grid[0, :]    # east values along cols (constant row)
    north_axis = north_grid[:, 0]   # north values along rows (constant col)

    # RectBivariateSpline requires strictly increasing axes.
    if east_axis[0] > east_axis[-1]:
        east_axis = east_axis[::-1]
        data = data[:, ::-1]

    if north_axis[0] > north_axis[-1]:
        north_axis = north_axis[::-1]
        data = data[::-1, :]

    return RectBivariateSpline(east_axis, north_axis, data.T)

In [10]:
elevation_data = load_elevation_data()
elevation_spline = make_elevation_spline(elevation_data, lat0, lon0, alt0)

In [9]:
def skew_symmetric(v):
  return np.array([
    [0, -v[2], v[1]],
    [v[2], 0, -v[0]],
    [-v[1], v[0], 0]
  ], dtype=np.float64)

# Matrix to select y, z compoenents of a vector
S = np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])

In [10]:
def heading_error(this, values, jacobians):
  pose = values.atPose3(this.keys()[0])
  v_w = values.atVector(this.keys()[1])
  R_vi = values.atRot3(this.keys()[2]).matrix() # rotation from imu frame to vehicle frame
  R_wi = pose.rotation().matrix() # rotation from imu frame to world frame
  
  v_i = R_wi.T @ v_w # Velocity in imu frame
  v_v = R_vi @ v_i # Velocity in vehicle frame
  
  speed = np.linalg.norm(v_w)
  v_target = np.array([speed, 0.0, 0.0])
  
  # We want the lateral and vertical velocity to be 0
  error = v_v - v_target
  
  if jacobians is not None:
    v_i_hat = skew_symmetric(v_i)
    J_pose = np.hstack((R_vi @ v_i_hat, np.zeros((3, 3))))
    
    J_Rvi = -R_vi @ v_i_hat
    
    J_target_vw = np.zeros((3, 3))
    J_target_vw[0, :] = v_w / (speed + 1e-8)
    J_vw = (R_vi @ R_wi.T) - J_target_vw
    
    jacobians[0] = J_pose
    jacobians[1] = J_vw
    jacobians[2] = J_Rvi
  return error

heading_constraint_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.5, 0.5, 0.5]))

In [11]:
def elevation_error(this, values, jacobians):
    pose = values.atPose3(this.keys()[0])
    offset = values.atVector(this.keys()[1])[0]
    t = pose.translation()
    east, north, z = t[0], t[1], t[2]
    error = np.array([z - elevation_spline.ev(east, north) - offset])

    if jacobians is not None:
        dz_de = elevation_spline.ev(east, north, dx=1)
        dz_dn = elevation_spline.ev(east, north, dy=1)
        # de/dt in world frame
        de_dt = np.array([-dz_de, -dz_dn, 1.0])

        R = pose.rotation().matrix()
        jacobians[0] = np.zeros((1, 6))
        jacobians[0][0, 3:6] = de_dt @ R
        # d(error)/d(offset)
        jacobians[1] = np.array([[-1.0]])

    return error

elevation_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([1.0]))

In [12]:
def terrain_normal(east, north):
    df_de = elevation_spline.ev(east, north, dx=1)  # d(elev)/d(east)
    df_dn = elevation_spline.ev(east, north, dy=1)  # d(elev)/d(north)
    n = np.array([-df_de, -df_dn, 1.0])
    return n / np.linalg.norm(n)


def normal_error(this, values, jacobians):
    pose = values.atPose3(this.keys()[0])
    R_vi = values.atRot3(this.keys()[1]).matrix()  # imu -> vehicle frame
    R_wi = pose.rotation().matrix()                # imu -> world frame

    t = pose.translation()
    normal = terrain_normal(t[0], t[1])

    c = R_vi.T @ np.array([0.0, 0.0, 1.0])
    up_world = R_wi @ c

    error = up_world - normal

    if jacobians is not None:
        J_rot = R_wi @ skew_symmetric(c)
        # wrt pose
        jacobians[0] = np.zeros((3, 6)) # ignores position
        jacobians[0][:, 0:3] = -J_rot
        # wrt the imu->vehicle extrinsic rotation R_vi
        jacobians[1] = J_rot

    return error

normal_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.3, 0.3, 0.3]))

In [13]:
idx = 5
roll, fit_file, racebox_file, is_c = roll_data[idx]
print(roll.id, is_c)
fit = load_fit_file(resolve_path(fit_file.file.uri))
fit_gps = get_gps_data(fit)
fit_graphs = get_fit_graph_data(fit)
racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])

start = {
  4: 170_000,
  3: 172_000,
  1: 57_000,
  6: 42_000,
  5: 44_000,
  9: 229_000
}[idx]
fit_gps = fit_gps.loc[start:]
fit_graphs['gps_data'] = fit_graphs['gps_data'].loc[start:]

38 False


In [14]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
accel_cal = calibration_data['accelerometer']
accel_raw, accel, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'x': 'accel_x', 'y': 'accel_y', 'z': 'accel_z'})
accel = accel[['x', 'y', 'z']]
# probably not really necessary, but there is a bunch of non white high frequency noise
accel = pd.DataFrame(lowpass_filter(accel.T, 5, 100).T, columns=['x', 'y', 'z'], index=accel.index)

gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'})
gyro = gyro[['x', 'y', 'z']] * (np.pi / 180) 
# gyro = pd.DataFrame(lowpass_filter(gyro.T, 5, 100).T, columns=['x', 'y', 'z'], index=gyro.index) 

magnet_cal = calibration_data['compass']
mag_raw, magnet, mag_fs = get_sensor_data(magnet_cal, fit['magnetometer_data_mesgs'], {'x': 'mag_x', 'y': 'mag_y', 'z': 'mag_z'})
magnet = magnet[['x', 'y', 'z']]

In [15]:
g_body = np.array(accel.loc[start-1000:start].mean())
m_body = np.array(magnet.loc[start-1000:start].mean())

up = -g_body / np.linalg.norm(g_body)
east = np.cross(m_body, up)
east = east / np.linalg.norm(east)
north = np.cross(up, east)
north = north / np.linalg.norm(north)
R_wi = np.vstack((east, north, up))
accel = accel * -9.81 # gtsam expects specific force in m/s^2, and the accelerometer data is in g's 

In [16]:
fit_enu = np.array(geodetic2enu(fit_gps.position_lat, fit_gps.position_long, fit_graphs['gps_data'].elevation, lat0, lon0, alt0)).T
fit_enu: pd.DataFrame = pd.DataFrame(fit_enu, index=fit_gps.index, columns=['x', 'y', 'z'])
fit_enu.z = elevation_spline.ev(fit_enu.x, fit_enu.y)

vel = np.array(fit_gps.velocity.to_list())
# project z direction of velocity to fit heightmap
dx = np.array([elevation_spline.ev(x, y, dx=1) for x, y in zip(fit_enu.x, fit_enu.y)])
dy = np.array([elevation_spline.ev(x, y, dy=1) for x, y in zip(fit_enu.x, fit_enu.y)])
vel[:, 2] = vel[:, 0] * dx + vel[:, 1] * dy

In [17]:
# # evalute spline in grid
# bounds = (fit_enu.x.min() - 10, fit_enu.x.max() + 10, fit_enu.y.min() - 10, fit_enu.y.max() + 10)
# x = np.linspace(bounds[0], bounds[1], 100)
# y = np.linspace(bounds[2], bounds[3], 100)
# z = elevation_spline(x, y).T

# # plot as 3d surface
# fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale='Viridis', opacity=0.8)])
# fig.add_trace(go.Scatter3d(x=fit_enu.x, y=fit_enu.y, z=fit_enu.z, mode='markers', name='fit GPS', marker=dict(size=2, color='red')))
# fig.update_layout(scene=dict(
#     xaxis_title='East (m)',
#     yaxis_title='North (m)',
#     zaxis_title='Elevation (m)',
#     aspectmode='manual',
#     aspectratio=dict(x=3, y=3, z=1)
# ), margin=dict(l=0, r=0, b=0, t=0))
# fig.show()

## Optimize

In [10]:
graph = gtsam.NonlinearFactorGraph()
estimate = gtsam.Values()

In [19]:
gps_base_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([3.0, 3.0, 5.0]))
doppler_base_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.5, 0.5, 1000.0]))
# gps_noise = gps_base_noise
# doppler_noise = doppler_base_noise
m_estimator = gtsam.noiseModel.mEstimator.Cauchy(2.0)
gps_noise = gtsam.noiseModel.Robust.Create(m_estimator, gps_base_noise)
doppler_noise = gtsam.noiseModel.Robust.Create(m_estimator, doppler_base_noise)

bias_rw_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([1e-3, 1e-3, 1e-3, 1e-3, 1e-3, 1e-3]) * 1e-1) # accel, gyro
imu_params = gtsam.PreintegrationParams.MakeSharedU(9.81)
imu_params.setAccelerometerCovariance(np.eye(3) * 1e-3)
imu_params.setGyroscopeCovariance(np.diag([3, 3, 1]) * 1e-3)
imu_params.setIntegrationCovariance(np.eye(3) * 1e-8)
current_bias = gtsam.imuBias.ConstantBias(g_body - (-up), np.array(gyro.mean()))
pim = gtsam.PreintegratedImuMeasurements(imu_params, current_bias)

In [20]:
start_pose = gtsam.Pose3(gtsam.Rot3(R_wi), gtsam.Point3(fit_enu.values[0]))
start_vel = gtsam.Point3(vel[0])
# Initial guess of roration from imu frame to vehicle frame
R_vi_start = gtsam.Rot3(np.array([
  [ 0.0, 1.0, 0.0],
  [-1.0, 0.0, 0.0],
  [ 0.0, 0.0, 1.0]
]))
if is_c:  R_vi_start = R_vi_start.compose(gtsam.Rot3.Yaw(np.deg2rad(180)))
offset_start = np.array([0.5])

graph.add(gtsam.PriorFactorPose3(X(0), start_pose, gtsam.noiseModel.Diagonal.Sigmas(np.array([5.0, 5.0, 5.0, 5.0, 5.0, 5.0]))))
graph.add(gtsam.PriorFactorPoint3(V(0), start_vel, gtsam.noiseModel.Diagonal.Sigmas(np.array([1.0, 1.0, 1.0]))))
graph.add(gtsam.PriorFactorConstantBias(B(0), current_bias, gtsam.noiseModel.Diagonal.Sigmas(np.array([1, 1, 1, 1, 1, 1]) * 1e-2)))
graph.add(gtsam.PriorFactorRot3(R(0), R_vi_start, gtsam.noiseModel.Diagonal.Sigmas(np.array([0.5, 0.5, 0.5]) * 1e3)))
graph.add(gtsam.PriorFactorVector(O(0), offset_start, gtsam.noiseModel.Diagonal.Sigmas(np.array([10.0]))))

estimate.insert(X(0), start_pose)
estimate.insert(V(0), start_vel)
estimate.insert(B(0), current_bias)
estimate.insert(R(0), R_vi_start)
estimate.insert(O(0), offset_start)

In [21]:
heading_cutoff = 0.3
rot_init = [start_pose.rotation()]
for i in range(1, fit_enu.shape[0] - 20):
  t_prev = fit_enu.index[i-1]
  t_curr = fit_enu.index[i]
  
  accel_data = accel.loc[t_prev:t_curr].values
  gyro_data = gyro.loc[t_prev:t_curr].values
  if accel_data.shape[0] == 0 or gyro_data.shape[0] == 0:
    print(f"Skipping index {i} due to missing IMU data")
    continue
  
  dt = 1.0 / 100.0
  for a, g in zip(accel_data, gyro_data):
    pim.integrateMeasurement(a, g, dt)
  
  graph.add(gtsam.ImuFactor(X(i-1), V(i-1), X(i), V(i), B(i-1), pim))
  graph.add(gtsam.BetweenFactorConstantBias(B(i-1), B(i), gtsam.imuBias.ConstantBias(), bias_rw_noise))
  graph.add(gtsam.GPSFactor(X(i), fit_enu.values[i], gps_noise)) #TODO: consider GPSFactorArmCalib
  graph.add(gtsam.PriorFactorVector(V(i), vel[i], doppler_noise))
  graph.add(gtsam.CustomFactor(elevation_noise, [X(i), O(0)], elevation_error))
  graph.add(gtsam.CustomFactor(normal_noise, [X(i), R(0)], normal_error))
  prev_pose = estimate.atPose3(X(i-1))
  
  speed = np.linalg.norm(vel[i])
  if speed > heading_cutoff: # only apply heading constraint when moving
    graph.add(gtsam.CustomFactor(heading_constraint_noise, [X(i), V(i), R(0)], heading_error))
    # use velocity for yaw and the terrain normal for pitch/roll
    yaw = np.arctan2(vel[i][1], vel[i][0]) # arctan2(east, north)
    # if is_c: yaw += np.deg2rad(180)
    
    up = terrain_normal(fit_enu.values[i-1][0], fit_enu.values[i-1][1])
    forward = np.array([np.cos(yaw), np.sin(yaw), 0.0]) # heading in the horizontal plane
    forward = forward - np.dot(forward, up) * up # project onto the terrain tangent plane
    forward = forward / np.linalg.norm(forward)
    left = np.cross(up, forward)
    R_wv_guess = gtsam.Rot3(np.column_stack([forward, left, up]))
    R_wi_guess = R_wv_guess.compose(R_vi_start)
    rot_init.append(R_wi_guess)
    prev_pose = gtsam.Pose3(R_wi_guess, fit_enu.values[i-1])
  else:
    rot_init.append(prev_pose.rotation())
    
  navstate = pim.predict(gtsam.NavState(prev_pose, vel[i-1]), current_bias)
  estimate.insert(X(i), navstate.pose())
  estimate.insert(V(i), navstate.velocity())
  estimate.insert(B(i), current_bias)
  
  pim.resetIntegrationAndSetBias(current_bias)

In [22]:
# params = gtsam.GncLMParams()
# params.setMaxIterations(1000)
# params.setVerbosityGNC(gtsam.GncLMParams.Verbosity.VALUES)
# optimizer = gtsam.GncLMOptimizer(graph, estimate, params)
params = gtsam.GaussNewtonParams()
params.setVerbosity('SUMMARY')
# params.
optimizer = gtsam.GaussNewtonOptimizer(graph, estimate, params)
# result = optimizer.optimize()
# params = gtsam.LevenbergMarquardtParams()
# params.setMaxIterations(1000)
# params.setVerbosityLM('VERBOSE')
# params.setLogFile(f'{DATA_PATH}/archive/tmp/{idx}_h.log')
# optimizer = gtsam.LevenbergMarquardtOptimizer(graph, estimate, params)
result = optimizer.optimize()

In [28]:
optimized_poses = []
optimized_vels = []
optimized_biases = []
for i in range(fit_enu.shape[0]):
  if result.exists(X(i)):
    optimized_poses.append(result.atPose3(X(i)))
    optimized_vels.append(result.atPoint3(V(i)))
    optimized_biases.append(result.atConstantBias(B(i)))
optimized_speeds = np.linalg.norm(np.array(optimized_vels)[:, :2], axis=1)

In [29]:
from gtsam import Marginals
marginals = Marginals(graph, result)
pose_marginals = []
velocity_marginals = []
speed_marginals = []
for i in range(fit_enu.shape[0]):
  if result.exists(X(i)):
    pose_marginals.append(marginals.marginalCovariance(X(i)))
    velocity_marginals.append(marginals.marginalCovariance(V(i)))
    speed_jacobian = np.zeros((1, 3))
    speed_jacobian[0, :2] = result.atPoint3(V(i))[:2] / optimized_speeds[i]
    speed_cov = speed_jacobian @ marginals.marginalCovariance(V(i)) @ speed_jacobian.T
    speed_marginals.append(speed_cov[0, 0])


## Analyze

In [30]:
racebox_enu = np.array(geodetic2enu(racebox_graph_data['gps_data'].lat, racebox_graph_data['gps_data'].long, racebox_graph_data['gps_data'].elevation, lat0, lon0, alt0)).T
racebox_enu: pd.DataFrame = pd.DataFrame(racebox_enu, index=racebox_graph_data['gps_data'].index, columns=['x', 'y', 'z'])

In [31]:
pose_arr = np.array([pose.translation() for pose in optimized_poses])
fig = go.Figure()
timestamps = fit_gps.index[:len(optimized_poses)]
fig.add_trace(go.Scatter(x=pose_arr[:, 0], y=pose_arr[:, 1], mode='markers', name='optimized', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=fit_enu.x, y=fit_enu.y, mode='markers', name='fit gps', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=racebox_enu.x, y=racebox_enu.y, mode='markers', name='racebox gps', hovertext=[f'timestamp: {ts}' for ts in racebox_enu.index]))
fig.show()

In [32]:
optimized_speeds = np.linalg.norm(np.array(optimized_vels)[:, :2], axis=1)
optimized_speeds = pd.Series(optimized_speeds, index=fit_gps.index[:len(optimized_vels)])
fig = go.Figure()
# plot speeds with error bands
fig.add_trace(go.Scatter(x=optimized_speeds.index, y=optimized_speeds.values, mode='lines', name='optimized speed'))
# fig.add_trace(go.Scatter(x=optimized_speeds.index, y=optimized_speeds.values, mode='lines', name='optimized speed'))

fig.add_trace(go.Scatter(x=racebox_graph_data['gps_data'].index + ((racebox_start_time - fit_start_time).total_seconds()) * 1000 + 400, y=racebox_graph_data['gps_data'].speed, mode='lines', name='racebox speed'))
fig.add_trace(go.Scatter(x=fit_gps.index , y=fit_gps.speed, mode='lines', name='fit gps speed'))
fig.add_trace(go.Scatter(x=list(optimized_speeds.index) + list(optimized_speeds.index)[::-1], y=list(optimized_speeds.values + 2 * np.sqrt(speed_marginals)) + list((optimized_speeds.values - 2 * np.sqrt(speed_marginals))[::-1]), fill='toself', fillcolor='rgba(0,100,80,0.2)', line=dict(color='rgba(255,255,255,0)')))
fig.show()

In [132]:
R_vi_est = result.atRot3(R(0))
yaws = [result.atPose3(X(i)).rotation().compose(R_vi_est).yaw() for i in range(len(optimized_poses))]
init_yaws = [rot.compose(R_vi_est).yaw() for rot in rot_init]
# px.line(pd.Series(yaws, index=fit_gps.index[:len(optimized_poses)]))
px.line(pd.DataFrame({'optimized': yaws, 'initial guess': init_yaws}, index=fit_gps.index[:len(optimized_poses)]))

In [133]:
# R_vi_est = result.atRot3(R(0))
yaws = [result.atPose3(X(i)).rotation().yaw() for i in range(len(optimized_poses))]
init_yaws = [rot.yaw() for rot in rot_init]
# px.line(pd.Series(yaws, index=fit_gps.index[:len(optimized_poses)]))
px.line(pd.DataFrame({'optimized': yaws, 'initial guess': init_yaws}, index=fit_gps.index[:len(optimized_poses)]))

In [134]:
pitchs = [result.atPose3(X(i)).rotation().compose(R_vi_est).pitch() for i in range(len(optimized_poses))]
init_pitchs = [rot.compose(R_vi_est).pitch() for rot in rot_init]
# px.line(pd.Series(yaws, index=fit_gps.index[:len(optimized_poses)]))
px.line(pd.DataFrame({'optimized': pitchs, 'initial guess': init_pitchs}, index=fit_gps.index[:len(optimized_poses)]))

In [137]:
vehicle_accel = []
world_accel = []
R_vi = result.atRot3(R(0)).matrix()
# R_vi = R_vi_start.matrix()
for i in range(1, fit_enu.shape[0] - 20):
  t_prev = fit_enu.index[i-1]
  t_curr = fit_enu.index[i]
  
  accel_data = accel.loc[t_prev:t_curr]
  # print(accel_data)
  gyro_data = gyro.loc[t_prev:t_curr]
  if accel_data.shape[0] == 0 or gyro_data.shape[0] == 0:
    print(f"Skipping index {i} due to missing IMU data")
    continue
  R_wi_est = result.atPose3(X(i-1)).rotation().matrix()
  bias = result.atConstantBias(B(i-1))
  accel_data = accel_data - bias.accelerometer()
  
  accel_v = accel_data.values @ R_vi.T
  accel_w = accel_data.values @ R_wi_est.T
  # accel_w = R_wi_est[:, [1] * len(accel_data)].T
  vehicle_accel.append(np.hstack([accel_data.index.values[:, None], accel_v]))
  world_accel.append(np.hstack([accel_data.index.values[:, None], accel_w]))

In [138]:
px.line(accel.loc[start:])

In [139]:
px.line(pd.DataFrame(np.vstack(vehicle_accel), columns=['timestamp', 'x', 'y', 'z']).set_index('timestamp'))

In [140]:
px.line(pd.DataFrame(np.vstack(world_accel), columns=['timestamp', 'x', 'y', 'z']).set_index('timestamp'))

In [251]:
# calcualte energy as 1/2v^2 + gh
optimized_energies = 1/2 * np.array([optimized_vels[i].dot(optimized_vels[i]) for i in range(len(optimized_vels))])
optimized_energies += np.array([optimized_poses[i].translation()[2] for i in range(len(optimized_poses))]) * 9.81
px.line(pd.Series(optimized_energies, index=fit_gps.index[:len(optimized_poses)]), title='Optimized Energy')

In [42]:
pos_speeds = np.linalg.norm(fit_enu.diff().iloc[1:].values, axis=1)
px.line(pd.Series(pos_speeds, index=fit_gps.index[1:]), title='Position Speed')

In [54]:
vels_sum = vel[:, 1].cumsum()
pos = fit_enu.y
# plot velocity cumsum against position
px.line(pd.DataFrame({'position': pos, 'velocity_cumsum': vels_sum/10}, index=fit_gps.index))

In [ ]:
px.line(np.)

In [ ]:
px.line(np.array([optimized_vels[i].dot(optimized_vels[i]) for i in range(len(optimized_vels))]))

In [ ]:
px.line(np.array([optimized_poses[i].translation()[2] for i in range(len(optimized_poses))]))

In [73]:
optimized_energies

array([2841.08304005, 2840.77492977, 2840.48777482, ..., 2900.10283214,
       2899.94256595, 2899.72222948], shape=(1772,))

In [ ]:
px.line(optimized_vels)

In [ ]:
px.line(vel)

In [ ]:
px.line(accel.loc[start:])

In [ ]:
px.line(np.array([bias.accelerometer() for bias in optimized_biases]))

In [10]:
# pan right/left, tilt up/down, roll clockwise/counterclockwise
fit = load_fit_file(f'{DATA_PATH}/archive/tmp/calibrate/2026-06-18-21-27-45.fit')

Caching archive_tmp_calibrate_2026-06-18-21-27-45.json


In [11]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'x': 'accel_x', 'y': 'accel_y', 'z': 'accel_z'})
accel_data = accel_data[['x', 'y', 'z']] * -9.81

In [ ]:
px.line(accel_data)

In [ ]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'})
gyro_data = gyro_data[['x', 'y', 'z']] * -(np.pi / 180) 
px.line(gyro_data)

In [ ]:
# # Vehicle should be oriented roughly in the direction of travel
# # TODO: make this aware of slip angles while turning

# def calc_heading_jacobians(R_vi, R_wi, v_i):
#   # Derivative with repect to pose
#   v_i_hat = skew_symmetric(v_i)
#   J_Rwi = S @ R_vi @ v_i_hat
  
#   J_pose = np.zeros((2, 6), dtype=np.float64)
#   J_pose[:, :3] = J_Rwi
  
#   # Derivative with repsect to world velocity
#   J_vw = S @ R_vi @ R_wi.T
  
#   # Derivative with respect to extrinsic rotation
#   J_Rvi = -S @ R_vi @ v_i_hat
  
#   return J_pose, J_vw, J_Rvi


# def heading_constraint(this, values, jacobians):
#   pose = values.atPose3(this.keys()[0])
#   v_w = values.atVector(this.keys()[1])
#   R_vi = values.atRot3(this.keys()[2]).matrix() # rotation from imu frame to vehicle frame
#   R_wi = pose.rotation().matrix() # rotation from imu frame to world frame
  
#   v_i = R_wi.T @ v_w # Velocity in imu frame
#   v_v = R_vi @ v_i # Velocity in vehicle frame
  
#   # We want the lateral and vertical velocity to be 0
#   error = S @ v_v
  
#   if jacobians is not None:
#     J_pose, J_vw, J_Rvi = calc_heading_jacobians(R_vi, R_wi, v_i)
#     jacobians[0] = J_pose
#     jacobians[1] = J_vw
#     jacobians[2] = J_Rvi
#   return error

# heading_constraint_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.5, 0.5]))

# Visual

In [5]:
roll_vid_data = []
for roll in files:
  if 'video_preview' in roll:
    roll_vid_data.append((roll['roll'], roll['fit'], roll['video_preview'], roll['racebox']))
roll_vid_data

[(Roll(id=44, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=3),
  RollFile(id=89, roll_id=44, file_id=37, local_start_ms=3488, local_end_ms=262595),
  RollFile(id=88, roll_id=44, file_id=63, local_start_ms=None, local_end_ms=None),
  RollFile(id=3444, roll_id=44, file_id=3169, local_start_ms=None, local_end_ms=None)),
 (Roll(id=1388, roll_date_id=154, buggy_id=3, driver_id=3, roll_number=6),
  RollFile(id=3255, roll_id=1388, file_id=3062, local_start_ms=3315, local_end_ms=260653),
  RollFile(id=3256, roll_id=1388, file_id=3063, local_start_ms=None, local_end_ms=None),
  RollFile(id=3450, roll_id=1388, file_id=3174, local_start_ms=None, local_end_ms=None)),
 (Roll(id=1387, roll_date_id=154, buggy_id=4, driver_id=2, roll_number=7),
  RollFile(id=3253, roll_id=1387, file_id=3060, local_start_ms=3457, local_end_ms=204606),
  RollFile(id=3254, roll_id=1387, file_id=3061, local_start_ms=None, local_end_ms=None),
  RollFile(id=3451, roll_id=1387, file_id=3175, local_start_ms=None, loca

In [6]:
idx = 1
roll, fit_file, vid, racebox_file = roll_vid_data[idx]
print(roll.id)
fit = load_fit_file(resolve_path(fit_file.file.uri))
fit_gps = get_gps_data(fit)
fit_graphs = get_fit_graph_data(fit)
racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])

fit_start_time = estimate_fit_timestamp(fit)
racebox_start_time = racebox_file.file.start_time
print(fit_start_time, racebox_start_time)

fig = go.Figure()
fig.add_trace(go.Scatter(x=racebox_graph_data['gps_data'].index + ((racebox_start_time - fit_start_time).total_seconds()) * 1000, y=racebox_graph_data['gps_data'].speed, mode='lines', name='racebox'))
fig.add_trace(go.Scatter(x=fit_gps.index , y=fit_gps.speed, mode='lines', name='fit'))
from lib.gpx import calculate_speed
calced_speed = calculate_speed(fit_gps)
fig.add_trace(go.Scatter(x=calced_speed.index, y=calced_speed, mode='lines', name='calculated speed'))
fig.show()

1388
2026-02-21 13:33:47.652000 2026-02-21 13:34:36.600000


In [11]:
racebox_enu = np.array(geodetic2enu(racebox_graph_data['gps_data'].lat, racebox_graph_data['gps_data'].long, racebox_graph_data['gps_data'].elevation, lat0, lon0, alt0)).T
racebox_enu: pd.DataFrame = pd.DataFrame(racebox_enu, index=racebox_graph_data['gps_data'].index, columns=['x', 'y', 'z'])
fit_enu = np.array(geodetic2enu(fit_gps.position_lat, fit_gps.position_long, fit_graphs['gps_data'].elevation, lat0, lon0, alt0)).T
fit_enu: pd.DataFrame = pd.DataFrame(fit_enu, index=fit_gps.index, columns=['x', 'y', 'z'])
# fit_enu.z = elevation_spline.ev(fit_enu.x, fit_enu.y)

vel = np.array(fit_gps.velocity.to_list())
# project z direction of velocity to fit heightmap
# dx = np.array([elevation_spline.ev(x, y, dx=1) for x, y in zip(fit_enu.x, fit_enu.y)])
# dy = np.array([elevation_spline.ev(x, y, dy=1) for x, y in zip(fit_enu.x, fit_enu.y)])
# vel[:, 2] = vel[:, 0] * dx + vel[:, 1] * dy

In [ ]:
# pose_arr = np.array([pose.translation() for pose in optimized_poses])
fig = go.Figure()
timestamps = fit_gps.index
# fig.add_trace(go.Scatter(x=pose_arr[:, 0], y=pose_arr[:, 1], mode='markers', name='optimized', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=fit_enu.x, y=fit_enu.y, mode='markers', name='fit gps', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=racebox_enu.x, y=racebox_enu.y, mode='markers', name='racebox gps', hovertext=[f'timestamp: {ts}' for ts in racebox_enu.index]))
fig.show()

In [12]:
roll_starts = {
  1: 55_000,
  3: 170_000,
  4: 41_000,
  6: 229_000,
  7: 106_000,
}
roll_start = roll_starts[idx]
vid_path = resolve_path(vid.file.uri)

## Video imu list

In [13]:
from scipy.signal import butter, filtfilt

def estimate_racebox_offset(virb_gyro, rb_gyro, offset_ms):
    """Extra ms to add to start_time-aligned racebox timestamps, from gyro envelope xcorr."""
    def envelope(t_ms, xyz):
        fs = 1000 / np.median(np.diff(t_ms))
        mag = np.linalg.norm(xyz, axis=1)
        b, a = butter(2, 6.0 / (fs / 2), 'low')
        return t_ms / 1000, filtfilt(b, a, np.abs(mag - np.median(mag)))

    vt, venv = envelope(virb_gyro.index.to_numpy(float), virb_gyro[['gyro_x', 'gyro_y', 'gyro_z']].to_numpy())
    rt, renv = envelope(rb_gyro.timestamp.to_numpy(float) + offset_ms, rb_gyro[['x', 'y', 'z']].to_numpy())
    if min(vt[-1], rt[-1]) - max(vt[0], rt[0]) < 30:
        return 0.0
    fs = 25.0
    grid = np.arange(max(vt[0], rt[0]) + 1, min(vt[-1], rt[-1]) - 1, 1 / fs)
    b, a = butter(2, [0.3 / (fs / 2), 6.0 / (fs / 2)], 'bandpass')
    v = filtfilt(b, a, np.interp(grid, vt, venv))
    r = filtfilt(b, a, np.interp(grid, rt, renv))
    v = (v - v.mean()) / (v.std() + 1e-12)
    r = (r - r.mean()) / (r.std() + 1e-12)
    n = int(10 * fs)
    c = np.correlate(v, r, 'full')[len(r) - 1 - n: len(r) + n]
    k = int(np.argmax(c))
    frac = 0.5 * (c[k - 1] - c[k + 1]) / (c[k - 1] - 2 * c[k] + c[k + 1]) if 0 < k < len(c) - 1 else 0.0
    return float((k - n + frac) / fs * 1000)

In [ ]:
import json
for idx, start in tqdm(roll_starts.items()):
    roll, fit_file, vid, racebox_file = roll_vid_data[idx]
    fit = load_fit_file(resolve_path(fit_file.file.uri))
    fit_gps = get_gps_data(fit)
    fit_graphs = get_fit_graph_data(fit)
    camera_start = get_camera_starts(fit)[0]
    camera_end = get_camera_ends(fit)[0]
    calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
    calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
    
    accel_cal = calibration_data['accelerometer']
    accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'acc_x': 'accel_x', 'acc_y': 'accel_y', 'acc_z': 'accel_z'})
    accel_data = accel_data[['acc_x', 'acc_y', 'acc_z']] * -9.81          # g -> specific force m/s^2
    accel_data = accel_data.loc[start:camera_end]
    
    gyro_cal = calibration_data['gyroscope']
    gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'gyro_x': 'gyro_x', 'gyro_y': 'gyro_y', 'gyro_z': 'gyro_z'})
    gyro_data = gyro_data[['gyro_x', 'gyro_y', 'gyro_z']] * -(np.pi / 180)   # deg/s -> rad/s
    gyro_full = gyro_data
    gyro_data = gyro_data.loc[start:camera_end]
    imu = pd.merge_asof(accel_data, gyro_data, on='timestamp')
    imu.timestamp = (imu.timestamp * 1e6).astype('int64')
    
    fit_enu = np.array(geodetic2enu(fit_gps.position_lat, fit_gps.position_long, fit_graphs['gps_data'].elevation, lat0, lon0, alt0)).T
    fit_enu: pd.DataFrame = pd.DataFrame(fit_enu, index=fit_gps.index, columns=['x', 'y', 'z'])
    fit_enu.z = elevation_spline.ev(fit_enu.x, fit_enu.y)

    vel = np.array(fit_gps.velocity.to_list())
    # project z direction of velocity to fit heightmap
    dx = np.array([elevation_spline.ev(x, y, dx=1) for x, y in zip(fit_enu.x, fit_enu.y)])
    dy = np.array([elevation_spline.ev(x, y, dy=1) for x, y in zip(fit_enu.x, fit_enu.y)])
    vel[:, 2] = vel[:, 0] * dx + vel[:, 1] * dy
    
    # use lat/long, but height from fit_enu
    gps_data = pd.DataFrame({'lat': fit_gps.position_lat, 'long': fit_gps.position_long, 'alt': fit_enu.z, 'timestamp': fit_gps.index * 1_000_000}, index=fit_gps.index)
    velocity = pd.DataFrame({'vx': vel[:, 0], 'vy': vel[:, 1], 'vz': vel[:, 2], 'timestamp': fit_gps.index * 1_000_000}, index=fit_gps.index)
    
    racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])
    
    fit_start_time = estimate_fit_timestamp(fit)
    racebox_start_time = racebox_file.file.start_time
    
    offset = (racebox_start_time - fit_start_time).total_seconds() * 1000
    racebox_graph_data['gps_data'].index += offset
    racebox_offset_ms = estimate_racebox_offset(gyro_full, racebox_graph_data['gyroscope'], offset)
    
    racebox_gps = pd.DataFrame({'lat': racebox_graph_data['gps_data'].lat, 'long': racebox_graph_data['gps_data'].long, 'alt': racebox_graph_data['gps_data'].elevation + 288.4, 'timestamp': racebox_graph_data['gps_data'].index * 1_000_000}, index=racebox_graph_data['gps_data'].index)
    racebox_speed = pd.DataFrame({'speed': racebox_graph_data['gps_data'].speed, 'timestamp': racebox_graph_data['gps_data'].index * 1_000_000}, index=racebox_graph_data['gps_data'].index)
    
    vid_path = vid.file.uri.replace('[[archive]]', 'archive').replace('[[videos]]', 'videos')
    data = dict(vid_path=vid_path, camera_start=camera_start * 1_000_000, racebox_offset_ms=racebox_offset_ms,
                imu_data=imu.to_dict(orient='records'), 
                gps_data=gps_data.to_dict(orient='records'), velocity=velocity.to_dict(orient='records'),
                racebox_gps=racebox_gps.to_dict(orient='records'), racebox_speed=racebox_speed.to_dict(orient='records'))
    print(data['camera_start'], imu.timestamp.min(), imu.timestamp.max(), f'racebox_offset_ms={racebox_offset_ms:+.0f}')
    
    with open(f'{DATA_PATH}/archive/vid_imu/{roll.id}.json', 'w') as f:
        json.dump(data, f)

  0%|          | 0/5 [00:00<?, ?it/s]

1388: racebox offset: 363.70 ms
3315000000 55005000000 260407000000 racebox_offset_ms=+364
37: racebox offset: 307.16 ms
3180000000 170000000000 350038000000 racebox_offset_ms=+307
38: racebox offset: 268.80 ms
3208000000 41008000000 211320000000 racebox_offset_ms=+269
1401: racebox offset: 339.60 ms
3572000000 229002000000 406622000000 racebox_offset_ms=+340
45: racebox offset: 284.83 ms
3555000000 106000000000 278779000000 racebox_offset_ms=+285


In [17]:
data['gps_data']

[{'lat': 40.441783759742975,
  'long': -79.9415799882263,
  'alt': 576.7321948106935,
  'timestamp': 99231000000},
 {'lat': 40.44178074225783,
  'long': -79.94158149696887,
  'alt': 576.732353983336,
  'timestamp': 99331000000},
 {'lat': 40.44177764095366,
  'long': -79.94158267043531,
  'alt': 576.7325733721261,
  'timestamp': 99431000000},
 {'lat': 40.44177470728755,
  'long': -79.94158376008272,
  'alt': 576.7324259534914,
  'timestamp': 99531000000},
 {'lat': 40.4417719412595,
  'long': -79.94158526882529,
  'alt': 576.7311610110917,
  'timestamp': 99631000000},
 {'lat': 40.44176917523146,
  'long': -79.94158711284399,
  'alt': 576.7295845945619,
  'timestamp': 99731000000},
 {'lat': 40.44176674447954,
  'long': -79.94158912450075,
  'alt': 576.7284676965744,
  'timestamp': 99831000000},
 {'lat': 40.44176397845149,
  'long': -79.94159046560526,
  'alt': 576.7307843204763,
  'timestamp': 99931000000},
 {'lat': 40.44176146388054,
  'long': -79.94159230962396,
  'alt': 576.73415361683